In [ ]:
import pandas as pd
import numpy as np
import os
from collections import Counter

In [ ]:
weekly_selections = pd.read_csv('./FFS_manager_weekly_selections.csv')
team_weekly_data = pd.read_csv('./fpl_team_weekly_data.csv')
player_data = pd.read_csv('./data/merged_player_data.csv')

In [ ]:
player_weekly_data = pd.read_csv('./fpl_player_weekly_data.csv')
managers = player_weekly_data[player_weekly_data['Position'] == 'Manager']
player_weekly_data = player_weekly_data[player_weekly_data['Position'] != 'Manager']
# managers#[['First Name', 'Last Name', 'Position', 'element']]

feats = [
    'element', 'fixture', 'opponent_team', 'total_points', 'was_home', 'team_h_score', 'team_a_score', 'round', 'minutes', 'goals_scored', 'assists',
    'clean_sheets', 'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards', 'red_cards', 'saves', 'bonus', 'bps', 'influence',
    'creativity', 'threat', 'ict_index', 'starts', 'expected_goals', 'expected_assists', 'expected_goal_involvements', 'expected_goals_conceded', 'value',
    'transfers_balance', 'selected', 'transfers_in', 'transfers_out', 'player_id', 'home_team_difficulty', 'away_team_difficulty', 'First Name',
    'Last Name', 'Team', 'Position',
]

player_weekly_data = player_weekly_data[feats]
player_weekly_data

## Add other data


In [ ]:
teams = pd.read_csv('./2024-25/teams.csv')
fpl_id_to_team_lookup = dict(zip(teams['id'], teams['name']))

player_weekly_data = player_weekly_data.rename(columns={
    'First Name': 'first_name', 'Last Name': 'last_name', 'Team': 'team', 'Position': 'position'})

# copy of player_weekly_data
player_weekly_data_copy = player_weekly_data.copy()

# Map opponent_team, h_team, and a_team to team IDs using the lookup dictionary
player_weekly_data_copy['opponent_team'] = player_weekly_data_copy['opponent_team'].map(fpl_id_to_team_lookup)

# Cater for loaned players by mapping their team IDs to the correct team names
# Cast both columns to object/string type to avoid dtype incompatibility
player_weekly_data_copy[['team', 'opponent_team']] = player_weekly_data_copy[['team', 'opponent_team']].astype('object')
# Update both columns simultaneously in a single operation
player_weekly_data_copy.loc[2489, ['team', 'opponent_team']] = ['Aston Villa', 'Man Utd'] # Marcus Rashford
player_weekly_data_copy.loc[2419, ['team', 'opponent_team']] = ['Aston Villa', 'Chelsea'] # Axel Disasi
player_weekly_data_copy.loc[7471, ['team', 'opponent_team']] = ['Crystal Palace', 'Chelsea'] # Trevor Chalobah
player_weekly_data_copy.loc[[8841, 8858], ['team', 'opponent_team']] = [['Crystal Palace', 'Chelsea'],['Crystal Palace', 'Chelsea']] # Ben Chilwell
player_weekly_data_copy.loc[10927, ['team', 'opponent_team']] = ['Everton', 'Southampton'] # Carlos Acaraz  Durán
player_weekly_data_copy.loc[[12440, 12457], ['team', 'opponent_team']] = [['Ipswich', 'Brighton'],['Ipswich', 'Brighton']] # Julio Enciso
player_weekly_data_copy.loc[13506, ['team', 'opponent_team']] = ['Ipswich', 'Aston Villa'] # Jaden Philogene
player_weekly_data_copy.loc[14867, ['team', 'opponent_team']] = ['Leicester', 'Spurs'] # Oliver Skipp
player_weekly_data_copy.loc[18729, ['team', 'opponent_team']] = ['Arsenal', 'Man Utd'] # Ayden Heaven
player_weekly_data_copy.loc[21607, ['team', 'opponent_team']] = ['Southampton', 'Newcastle'] # Ryan Fraser
player_weekly_data_copy.loc[24551, ['team', 'opponent_team']] = ['West Ham', 'Brighton'] # Evan Ferguson
player_weekly_data_copy.loc[25304, ['team', 'opponent_team']] = ["Nott'm Forest", 'West Ham'] # James Ward-Prowse


player_weekly_data_copy.loc[(player_weekly_data_copy['fixture'] == 123) & (player_weekly_data_copy['team'] == 'Aston Villa'), 'opponent_team'] = 'Chelsea' # Give Aston Villa an opponent in fixture 13
player_weekly_data_copy.loc[(player_weekly_data_copy['fixture'] == 62) & (player_weekly_data_copy['team'] == 'Aston Villa'), 'opponent_team'] = 'Man Utd'
player_weekly_data_copy.loc[(player_weekly_data_copy['fixture'] == 194) & (player_weekly_data_copy['team'] == 'Aston Villa'), 'opponent_team'] = 'Leicester'
player_weekly_data_copy.loc[(player_weekly_data_copy['fixture'] == 23) & (player_weekly_data_copy['team'] == 'Crystal Palace'), 'opponent_team'] = 'Chelsea'
player_weekly_data_copy.loc[(player_weekly_data_copy['fixture'] == 194) & (player_weekly_data_copy['team'] == 'Crystal Palace'), 'opponent_team'] = 'Chelsea'
player_weekly_data_copy.loc[(player_weekly_data_copy['fixture'] == 194) & (player_weekly_data_copy['team'] == 'Chelsea'), 'opponent_team'] = 'Crystal Palace'
player_weekly_data_copy.loc[(player_weekly_data_copy['fixture'] == 98) & (player_weekly_data_copy['team'] == 'Everton'), 'opponent_team'] = 'Southampton'
player_weekly_data_copy.loc[(player_weekly_data_copy['fixture'] == 97) & (player_weekly_data_copy['team'] == 'West Ham'), 'opponent_team'] = "Nott'm Forest"
player_weekly_data_copy.loc[(player_weekly_data_copy['fixture'] == 33) & (player_weekly_data_copy['team'] == 'Ipswich'), 'opponent_team'] = 'Brighton'
player_weekly_data_copy.loc[(player_weekly_data_copy['fixture'] == 204) & (player_weekly_data_copy['team'] == 'Ipswich'), 'opponent_team'] = 'Brighton'
player_weekly_data_copy.loc[(player_weekly_data_copy['fixture'] == 56) & (player_weekly_data_copy['team'] == 'Ipswich'), 'opponent_team'] = 'Aston Villa'
player_weekly_data_copy.loc[(player_weekly_data_copy['fixture'] == 10) & (player_weekly_data_copy['team'] == 'Leicester'), 'opponent_team'] = 'Tottenham'
player_weekly_data_copy.loc[(player_weekly_data_copy['fixture'] == 5) & (player_weekly_data_copy['team'] == 'Southampton'), 'opponent_team'] = 'Newcastle'
player_weekly_data_copy.loc[(player_weekly_data_copy['fixture'] == 131) & (player_weekly_data_copy['team'] == 'Man Utd'), 'opponent_team'] = 'Arsenal'
player_weekly_data_copy.loc[(player_weekly_data_copy['fixture'] == 170) & (player_weekly_data_copy['team'] == 'West Ham'), 'opponent_team'] = 'Brighton'

player_weekly_data_copy[player_weekly_data_copy['opponent_team'] == player_weekly_data_copy['team']][['fixture', 'team', 'opponent_team', 'round']]

In [ ]:
# If was_home is True, set team to the home team; otherwise, set it to the away team
player_weekly_data_copy['h_team'] = np.where(player_weekly_data_copy['was_home'], player_weekly_data_copy['team'], player_weekly_data_copy['opponent_team'])
player_weekly_data_copy['a_team'] = np.where(player_weekly_data_copy['was_home'], player_weekly_data_copy['opponent_team'], player_weekly_data_copy['team'])

# Change home_team_difficulty and away_team_difficulty to team_difficult and opponent_team_difficulty respectively
player_weekly_data_copy['difficulty'] = np.where(player_weekly_data_copy['was_home'], player_weekly_data_copy['home_team_difficulty'], player_weekly_data_copy['away_team_difficulty'])
player_weekly_data_copy['opponent_difficulty'] = np.where(player_weekly_data_copy['was_home'], player_weekly_data_copy['away_team_difficulty'], player_weekly_data_copy['home_team_difficulty'])



player_weekly_data_copy[['was_home', 'team', 'opponent_team', 'h_team', 'a_team', 'round']]

### Add Odds


In [ ]:
odds = pd.read_csv('./E0 24-25.csv')
odds

In [ ]:
odds_to_fpl_team_name_mapping = {
    'Arsenal': 'Arsenal',
    'Aston Villa': 'Aston Villa',
    'Bournemouth': 'Bournemouth',
    'Brentford': 'Brentford',
    'Brighton': 'Brighton',
    'Chelsea': 'Chelsea',
    'Crystal Palace': 'Crystal Palace',
    'Everton': 'Everton',
    'Fulham': 'Fulham',
    'Ipswich': 'Ipswich',
    'Leicester': 'Leicester',
    'Liverpool': 'Liverpool',
    'Manchester City': 'Man City',
    'Manchester United': 'Man Utd',
    'Newcastle United': 'Newcastle',
    "Nottingham Forest": "Nott'm Forest",
    'Southampton': 'Southampton',
    'Tottenham': 'Spurs',
    'West Ham': 'West Ham',
    'Wolverhampton Wanderers': 'Wolves'
}

# Copy of odds DataFrame to avoid modifying the original
odds_copy = odds.copy()

# Map the team names in the odds DataFrame to FPL team names using the mapping dictionary
odds_copy['h_team'] = odds_copy['h_team'].map(odds_to_fpl_team_name_mapping)
odds_copy['a_team'] = odds_copy['a_team'].map(odds_to_fpl_team_name_mapping)
odds_copy

In [ ]:
# Merge the odds DataFrame with player_weekly_data_copy on h_team, a_team
player_weekly_data_odds = pd.merge(
    player_weekly_data_copy,
    odds_copy[['h_team', 'a_team', 'B365H', 'B365D', 'B365A']],
    on=['h_team', 'a_team'],
    how='left'
)

player_weekly_data_odds[['h_team', 'a_team', 'B365H', 'B365D', 'B365A']]

In [ ]:
# Extract the relevant data values
# Convert the data to probabilities

def odds_to_probabilities(row):
    WHH = round(1/row['B365H'], 5)
    WHD = round(1/row['B365D'], 5)
    WHA = round(1/row['B365A'], 5)

    # Normalize the probabilities (to make the probabilities sum to 100%)
    WH_sum = WHH + WHD + WHA
    WHH_ = round(WHH/WH_sum, 3)
    WHD_ = round(WHD/WH_sum, 3)
    WHA_ = round(WHA/WH_sum, 3)

    return pd.Series([WHH_, WHD_, WHA_], index=['home_win_prob', 'draw_prob', 'away_win_prob'])

odds_prob_cols  = player_weekly_data_odds.apply(odds_to_probabilities, axis=1)
odds_prob_cols.columns = ['home_win_prob', 'draw_prob', 'away_win_prob']

# 2. Join the original DataFrame with the new columns all at once
player_weekly_data_probs = pd.concat([player_weekly_data_odds, odds_prob_cols], axis=1)

player_weekly_data_probs['win_prob'] = np.where(player_weekly_data_probs['was_home'], player_weekly_data_probs['home_win_prob'], player_weekly_data_probs['away_win_prob'])
player_weekly_data_probs['draw_prob'] = player_weekly_data_probs['draw_prob']
player_weekly_data_probs['lose_prob'] = np.where(player_weekly_data_probs['was_home'], player_weekly_data_probs['away_win_prob'], player_weekly_data_probs['home_win_prob'])

player_weekly_data_probs#[['was_home','home_win_prob', 'draw_prob', 'away_win_prob', 'win_prob', 'draw_prob', 'lose_prob']]

### Percentage Net Transfer


In [ ]:
def ownership_change(row):
    net_transfers = row['transfers_in'] - row['transfers_out']
    total_transfers = row['transfers_in'] + row['transfers_out']
    net_transfers_pct = net_transfers / total_transfers if total_transfers != 0 else 0

    return net_transfers_pct

player_weekly_data_probs['percentage_net_transfers'] = player_weekly_data_probs.apply(ownership_change, axis=1)

player_weekly_data_probs = player_weekly_data_probs.drop_duplicates(subset=['element', 'round'], keep='first')

player_weekly_data_probs

### Add xP


In [ ]:
xP_data = pd.read_csv(f'./data/vaastav/data/2024-25/gws/gw1.csv')
for i in range(2,39):
    xP_data = pd.concat([xP_data, pd.read_csv(f'./data/vaastav/data/2024-25/gws/gw{i}.csv')], ignore_index=True)

xP_data = xP_data.drop_duplicates(subset=['element', 'round'], keep='first')
xP_data

In [ ]:
player_data_xP = player_weekly_data_probs.merge(xP_data[['name', 'xP', 'element', 'round' ]], on=['element', 'round'], how='left')
player_data_xP

In [ ]:
# Fill xP col rounds with only 0s
missing_xp_gws = [22, 32, 34]

url = "https://raw.githubusercontent.com/olbauday/FPL-Core-Insights/refs/heads/main/data/2025-2026/By%20Gameweek"

frames = []

for gw in missing_xp_gws:
    data = pd.read_csv(f'{url}/GW{gw}/player_gameweek_stats.csv')

    xP_temp = data[['id', 'ep_this']].copy()
    xP_temp['round'] = gw

    frames.append(xP_temp)

xP = pd.concat(frames, ignore_index=True)


xP = xP.rename(columns={'id': 'element', 'ep_this': 'xP_new'})

player_data_xP = player_data_xP.merge(
    xP,
    on=['element', 'round'],
    how='left'
)

mask = player_data_xP['round'].isin([22, 32, 34])

player_data_xP.loc[mask, 'xP'] = player_data_xP.loc[mask, 'xP_new']

player_data_xP = player_data_xP.drop(columns='xP_new')
player_data_xP

### Add CBITR


In [ ]:
# Get team match stats
olbauday_team_data = pd.read_csv('https://raw.githubusercontent.com/olbauday/FPL-Core-Insights/refs/heads/main/data/2024-2025/matches/matches.csv')
olbauday_teams = pd.read_csv('https://raw.githubusercontent.com/olbauday/FPL-Core-Insights/refs/heads/main/data/2024-2025/teams/teams.csv')

# olbauday_team_data_23 = pd.read_csv('https://raw.githubusercontent.com/olbauday/FPL-Core-Insights/refs/heads/main/data/2023-2024/matches/matches.csv')
# olbauday_teams_23 = pd.read_csv('https://raw.githubusercontent.com/olbauday/FPL-Core-Insights/refs/heads/main/data/2023-2024/teams/teams.csv')



In [ ]:


team_code_to_id = dict(zip(olbauday_teams['code'], olbauday_teams['id']))

olbauday_team_data['home_team'] = olbauday_team_data['home_team'].map(team_code_to_id)
olbauday_team_data['home_team'] = olbauday_team_data['home_team'].map(fpl_id_to_team_lookup)

olbauday_team_data['away_team'] = olbauday_team_data['away_team'].map(team_code_to_id)
olbauday_team_data['away_team'] = olbauday_team_data['away_team'].map(fpl_id_to_team_lookup)

olbauday_team_data =  olbauday_team_data.rename(columns={'gameweek': 'round'})


olbauday_team_data["team"] = olbauday_team_data["home_team"]
olbauday_team_data["opponent_team"] = olbauday_team_data["away_team"]
olbauday_team_data["team_elo"] = olbauday_team_data["home_team_elo"]
olbauday_team_data["opponent_elo"] = olbauday_team_data["away_team_elo"]
olbauday_team_data["team_xG"] = olbauday_team_data["home_expected_goals_xg"]
olbauday_team_data["opponent_xG"] = olbauday_team_data["away_expected_goals_xg"]
olbauday_team_data["team_xGOT"] = olbauday_team_data["home_xg_on_target_xgot"]
olbauday_team_data["opponent_xGOT"] = olbauday_team_data["away_xg_on_target_xgot"]  # Target quality faced by GK
olbauday_team_data["shots_faced"] = olbauday_team_data["home_shots_on_target"]
olbauday_team_data["opponent_shots_faced"] = olbauday_team_data["away_shots_on_target"]
olbauday_team_data["big_chances_faced"] = olbauday_team_data["home_big_chances"]
olbauday_team_data["opponent_big_chances_faced"] = olbauday_team_data["away_big_chances"]

olbauday_team_data[['round', 'team', 'opponent_team', 'team_elo', 'opponent_elo', 'team_xG', 'opponent_xG', 'team_xGOT', 'opponent_xGOT', 'shots_faced',
                    'opponent_shots_faced', 'big_chances_faced', 'opponent_big_chances_faced']].ffill()


In [ ]:
# Get the player match stats
olbauday_url = "https://raw.githubusercontent.com/olbauday/FPL-Core-Insights/refs/heads/main/data/2024-2025/playermatchstats"
olbauday_data = pd.read_csv(f'{olbauday_url}/GW1/playermatchstats.csv')
olbauday_data['round'] = 1

for gw in range(2,39):
    gw_data = pd.read_csv(f'{olbauday_url}/GW{gw}/playermatchstats.csv')
    gw_data['round'] = gw

    olbauday_data = pd.concat([olbauday_data, gw_data], ignore_index=True)

olbauday_data = olbauday_data.rename(columns={'player_id': 'element'}).drop_duplicates(subset=['element', 'round'], keep='first')
olbauday_data.to_csv('olbauday_data_playermatchstats.csv', index=False)
olbauday_data

In [ ]:
cols_to_add = [
    'element', 'interceptions', 'recoveries', 'blocks', 'clearances', 'tackles', 'total_shots', 'shots_on_target', 'successful_dribbles',
    'touches_opposition_box', 'touches', 'chances_created', 'goals_prevented', 'sweeper_actions', 'accurate_passes_percent', 'successful_dribbles_percent',
    'tackles_won_percent', 'round'    'tackles_won_percent', 'round'

]

player_data_cbitr = player_data_xP.merge(olbauday_data[cols_to_add], on=['element', 'round'], how='left')

player_data_cbitr[cols_to_add] = player_data_cbitr[cols_to_add].fillna(0)
player_data_cbitr

In [ ]:
player_data_cbitr = player_data_cbitr.merge(
            olbauday_team_data[['team', 'opponent_team', 'team_elo', 'opponent_elo', 'team_xG', 'opponent_xG', 'team_xGOT', 'opponent_xGOT', 'shots_faced',
                    'opponent_shots_faced', 'big_chances_faced', 'opponent_big_chances_faced']], on=['team', 'opponent_team'], how='left')

player_data_cbitr["elo_diff"] = player_data_cbitr["team_elo"] - player_data_cbitr["opponent_elo"]
player_data_cbitr

### Per 90


In [ ]:
feats_90 = [
    'total_points', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded','saves',  'expected_goals', 'expected_assists', 'expected_goal_involvements',
    'expected_goals_conceded', 'interceptions', 'recoveries', 'blocks', 'clearances', 'tackles', 'total_shots', 'chances_created', 'goals_prevented',
    'sweeper_actions', 'tackles_won_percent'
    ]

for col in feats_90:
        per_90_col_name = f"{col}_per_90"

        # Safe division vectorized with numpy
        player_data_cbitr[per_90_col_name] = (player_data_cbitr[col] / player_data_cbitr['minutes']) * 90.0#, np.nan


player_data_cbitr[[f"{feat}_per_90" for feat in feats_90]] = player_data_cbitr[[f"{feat}_per_90" for feat in feats_90]].fillna(0)
player_data_cbitr
    # if drop_original:
    #     player_data_cbitr.drop(columns=feats_90, inplace=True)

### Roll Data


In [ ]:
rolling_data = player_data_cbitr.copy()

## Cater for the missing rounds before rolling
# 1. Get all unique players and all unique rounds
all_players = rolling_data['element'].unique()
all_rounds = range(rolling_data['round'].min(), rolling_data['round'].max() + 1)

# 2. Create a MultiIndex of every player x every round
multi_idx = pd.MultiIndex.from_product([all_players, all_rounds], names=['element', 'round'])

# 3. Reindex the dataframe
# This inserts "empty" rows for missing player/round combinations
rolling_data = rolling_data.set_index(['element', 'round']).reindex(multi_idx).reset_index()

# Fill statistical columns with 0
rolling_data = rolling_data.fillna(0)

# Sort to ensure chronological order for the rolling window
rolling_data = rolling_data.sort_values(['element', 'round'])

rolling_data

In [ ]:
# 1. Calculate the maximum 'selected' value across all players for each round
max_per_round = rolling_data.groupby('round')['selected'].max()

# 2. Shift the values down by 1 round (so Round 2 gets the max of Round 1, etc.)
prev_round_max = max_per_round.shift(1)
prev_round_total = np.floor(prev_round_max/0.80)
prev_round_total
# # 3. Map these 'previous round max' values back to the original DataFrame
rolling_data['total_managers'] = rolling_data['round'].map(prev_round_total)

rolling_data['selected_by_percent'] = (rolling_data['selected'] / rolling_data['total_managers'])
rolling_data['ownership_change'] = rolling_data.groupby('element')['selected'].diff()
rolling_data['percentage_net_transfers'] = rolling_data.groupby('element')['selected_by_percent'].diff()

In [ ]:
rolling_data['points'] = rolling_data['total_points'] - rolling_data['bonus']
rolling_data

In [ ]:
rolling_features = [
    'points', 'xP',  'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'own_goals', 'yellow_cards', 'saves', 'influence',
    'creativity', 'threat', 'ict_index', 'starts', 'expected_goals', 'expected_assists', 'expected_goal_involvements', 'expected_goals_conceded',
    'interceptions', 'recoveries', 'blocks', 'clearances', 'tackles', 'total_shots', 'chances_created', 'goals_prevented', 'sweeper_actions',
    'tackles_won_percent',

    'team_xG', 'opponent_xG', 'team_xGOT', 'opponent_xGOT', 'shots_faced', 'opponent_shots_faced', 'big_chances_faced', 'opponent_big_chances_faced'
] + [f"{feat}_per_90" for feat in feats_90]


SPANS = [1,3,5]

# Sort to ensure chronological order for the rolling window
data_to_roll = rolling_data.sort_values(['element', 'round']).reset_index(drop=True)


#### Simple Averages


In [ ]:
# data_to_roll = data_to_roll[data_to_roll['team'] != 'Mainz 05']
# Averagae values
new_features = {}
for col in rolling_features:
    # The rolling window now spans actual calendar rounds
    grp = data_to_roll.groupby('element')[col]

    for span in SPANS:
        new_features[f'{col}_rolling_{span}'] = grp.transform(
            lambda x: x.shift(1).rolling(span, min_periods=1).mean()
        )

# Join all new columns at once (no fragmentation)
data_to_roll = pd.concat(
    [data_to_roll, pd.DataFrame(new_features)],
    axis=1
)

data_to_roll

#### Weighted Averages


In [ ]:
ewm_features = {}
grouped = data_to_roll.groupby('element', sort=False)

# for col  in EWMA_SPANS:rollerolled_datad_datarolled_data
for col in rolling_features:
    shifted = grouped[col].shift(1)
    for span in SPANS:
        ewm_features[f'{col}_{span}_ewm'] = (
            shifted.groupby(data_to_roll['element'])
            .ewm(span=span, adjust=False)
            .mean()
            .reset_index(level=0, drop=True)
        )

data_to_roll = pd.concat([data_to_roll, pd.DataFrame(ewm_features)], axis=1)

# Remove fillna(0) values that were added
rolled_data = data_to_roll[data_to_roll['position'] != 0]
rolled_data.to_csv('./rolled_data_24_25.csv', index=False)
rolled_data

## Organize Manager Data


In [ ]:
player_data = pd.read_csv('./rolled_data_24_25.csv')
player_data[player_data['round'] == 38]

In [ ]:
selections_df = (
    weekly_selections.groupby(['team_id', 'game_week'])['element']
    .apply(list)
    .reset_index(name='squad')
    .rename(columns={'game_week': 'round'})
)

selections_df

In [ ]:
# 1. Create a fast lookup dictionary (element ID -> Position)
element_pos_map = (
    player_data[['element', 'position']]
    .drop_duplicates(subset=['element'])
    .set_index('element')['position']
    .to_dict()
)

# 2. Extract position lists cleanly
positions = ['Midfielder', 'Forward', 'Defender', 'Goalkeeper']

for pos in positions:
    col_name = pos.lower() + 's'
    selections_df[col_name] = selections_df['squad'].apply(
        lambda squad: [player for player in squad if element_pos_map.get(player) == pos]
    )

selections_df

In [ ]:
# positions_players

In [ ]:
# Form a binary representation of selections for each position
positions = ['goalkeeper', 'defender', 'midfielder', 'forward']

gw_position_players = {}
for round in range(1, 39):
    # if round == 1:
    curr_player_data = player_data[player_data['round'] ==  round]
    # else:
    #     curr_player_data = player_data[player_data['round'] < round+1]

    positions_players = {}
    for p in positions:
        positions_players[p] = curr_player_data[curr_player_data['position'].str.lower() == p][['element']].drop_duplicates(subset=['element'])['element'].tolist()

    gw_position_players[round] = positions_players

# gw_position_players

In [ ]:
players_by_round = player_weekly_data.groupby('round')['element'].apply(list).to_dict()

def weekly_selections_binary(row, pos):
    # print(row[pos.lower() + 's'])
    gw = row['round']

    positions_players = gw_position_players[gw]
    squad_set = set(row[pos.lower() + 's'])
    round_players = positions_players[pos]

    return [1 if player in squad_set else 0 for player in round_players]

for pos in positions_players.keys():
    selections_df[pos + '_binary'] = selections_df.apply(lambda row: weekly_selections_binary(row, pos), axis=1)

selections_df

In [ ]:
league_1 = pd.read_csv('./FFS_League_1.csv')
league_teams = league_1.loc[0:19,].copy()
league_team_ids = league_teams['Team id'].tolist()
league_selections_df = selections_df[selections_df['team_id'].isin(league_team_ids)].copy()
league_selections_df

In [ ]:
league_selections_df.to_csv('./league_selections_df.csv', index=False)

## League Selection Probabilities

This is $p_{j,t}$ from the cross-sectional league baseline (see the "Cross-Sectional
League Baseline" derivation in the write-up): for each position $j$ and game week $t$,
the fraction of the H2H league's own selection "slots" that went to each player, pooled
across all 20 managers in `league_selections_df` -- not FPL's global `selected_by_percent`,
which is computed across all ~11M FPL players and says nothing about this specific
league's behavior.

Reuses the `{position}_binary` columns already built above (one 0/1 per player in that
position's round-`gw_position_players[round][pos]` pool, per manager): summing them across
the 20 managers gives each player's raw selection count that round, and dividing by the
round-position total (which is exactly `20 managers * squad size for that position`, since
every manager contributes a full squad) turns it into a probability distribution over that
position's player pool for that round -- this is the discrete population the Dirichlet
regression prior treats as `Dir(alpha_league)`-distributed. Rounds/positions with no
recorded league squads (should not happen for 1-38 given `league_selections_df` covers the
full season) are simply skipped, leaving any such player-round at 0 rather than raising.


In [ ]:
league_select_probs_map = {}  # (round, element) -> probability, within that round's position pool

for rnd in range(1, 39):
    if rnd not in gw_position_players:
        continue
    round_rows = league_selections_df[league_selections_df['round'] == rnd]
    if round_rows.empty:
        continue

    for pos in positions:
        round_players = gw_position_players[rnd][pos]
        if not round_players:
            continue

        # one row per manager, one column per player in `round_players` (same order
        # weekly_selections_binary built each manager's binary vector in)
        binary_matrix = np.array(round_rows[pos + '_binary'].tolist())
        counts = binary_matrix.sum(axis=0)
        total = counts.sum()
        probs = counts / total if total > 0 else np.zeros_like(counts, dtype=float)

        for player, p in zip(round_players, probs):
            league_select_probs_map[(rnd, player)] = p

player_data['league_select_probs'] = player_data.apply(
    lambda row: league_select_probs_map.get((row['round'], row['element']), 0.0), axis=1
)

player_data[['round', 'element', 'position', 'league_select_probs']]

In [ ]:
# Sanity check: each round-position's probabilities should sum to ~1 (0 only for a
# round/position with no recorded league squads at all, which shouldn't happen here)
check = (
    player_data.groupby(['round', 'position'])['league_select_probs'].sum()
)
print(f"round-position sums outside [0.99, 1.01] (excluding exact 0): "
      f"{((check < 0.99) & (check != 0)).sum() + (check > 1.01).sum()} / {len(check)}")

player_data.to_csv('./rolled_data_24_25.csv', index=False)